# Notebook 02: Linear Models

## Why This Matters

Linear models are the workhorses of production ML: fast, interpretable, reliable, and surprisingly competitive in high-dimensional spaces. Logistic regression is still used for ad CTR prediction at Facebook scale. Ridge regression underlies collaborative filtering. Lasso is used for feature selection in genomics with p >> n problems. Despite the enthusiasm for deep learning, a senior ML engineer reaches for a linear model first and justifies any complexity increase. Understanding regularization and when linear models win is foundational for every more complex method.

In [ ]:
%pip install -q scikit-learn numpy pandas matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_regression, make_classification, load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
np.random.seed(42)
print("Ready.")

## 1. OLS vs Regularized Regression

**OLS**: minimizes `||Xw - y||^2`. Closed-form: `w = (X^T X)^-1 X^T y`.

**When OLS breaks**:
- p > n: `X^T X` is singular (not invertible)
- Highly correlated features: near-singular -> unstable coefficients

**Ridge (L2)**: adds `lambda * ||w||^2` to loss - shrinks all coefficients toward zero proportionally. Never zeroes out coefficients. Handles correlated features well.

**Lasso (L1)**: adds `lambda * ||w||_1` to loss - sets some coefficients to exactly zero (sparse solution). Automatic feature selection. Unstable with correlated features.

**ElasticNet**: L1 + L2 mix - useful when features are correlated and you want sparsity.

In [ ]:
class RidgeScratch:
    def __init__(self, alpha=1.0, lr=0.01, n_iter=1000):
        self.alpha = alpha; self.lr = lr; self.n_iter = n_iter
    
    def fit(self, X, y):
        n, p = X.shape; self.w_ = np.zeros(p); self.b_ = 0.0
        self.loss_history_ = []
        for _ in range(self.n_iter):
            preds = X @ self.w_ + self.b_; residuals = preds - y
            grad_w = (2/n) * X.T @ residuals + 2 * self.alpha * self.w_
            grad_b = (2/n) * residuals.sum()
            self.w_ -= self.lr * grad_w; self.b_ -= self.lr * grad_b
            self.loss_history_.append((residuals**2).mean() + self.alpha*(self.w_**2).sum())
        return self
    
    def predict(self, X): return X @ self.w_ + self.b_

X, y = make_regression(n_samples=500, n_features=20, n_informative=10, noise=20, random_state=42)
scaler = StandardScaler(); X_s = scaler.fit_transform(X)
X_tr, X_te, y_tr, y_te = train_test_split(X_s, y, test_size=0.2, random_state=42)

ridge_scratch = RidgeScratch(alpha=1.0, lr=0.01).fit(X_tr, y_tr)
ridge_sklearn = Ridge(alpha=1.0).fit(X_tr, y_tr)

rmse_scratch = np.sqrt(mean_squared_error(y_te, ridge_scratch.predict(X_te)))
rmse_sklearn = np.sqrt(mean_squared_error(y_te, ridge_sklearn.predict(X_te)))
print(f"From-scratch Ridge RMSE: {rmse_scratch:.4f}")
print(f"sklearn Ridge RMSE:      {rmse_sklearn:.4f}")
print("(Difference due to optimization method - sklearn uses closed-form)")

## 2. Ridge vs Lasso: Regularization Paths

**Geometric intuition**: 
- L2 constraint region is a sphere (smooth, no corners) - gradient descent slides along the surface and rarely hits an axis -> coefficients shrink but rarely zero
- L1 constraint region is a diamond (pointy corners on axes) - gradient descent frequently terminates at a corner where one coordinate = 0 -> sparsity

In [ ]:
class LassoScratch:
    def __init__(self, alpha=1.0, n_iter=1000, tol=1e-4):
        self.alpha = alpha; self.n_iter = n_iter; self.tol = tol
    
    def _soft_threshold(self, rho, alpha):
        if rho < -alpha: return rho + alpha
        elif rho > alpha: return rho - alpha
        else: return 0.0  # ZERO: this creates sparsity
    
    def fit(self, X, y):
        n, p = X.shape; self.w_ = np.zeros(p)
        for iteration in range(self.n_iter):
            w_old = self.w_.copy()
            for j in range(p):
                residual = y - X @ self.w_ + X[:, j] * self.w_[j]
                rho = X[:, j] @ residual / n
                self.w_[j] = self._soft_threshold(rho, self.alpha)
            if np.max(np.abs(self.w_ - w_old)) < self.tol: break
        return self
    
    def predict(self, X): return X @ self.w_

alphas = np.logspace(-3, 2, 50)
ridge_coefs = [Ridge(alpha=a).fit(X_tr, y_tr).coef_ for a in alphas]
lasso_coefs = [Lasso(alpha=a, max_iter=5000).fit(X_tr, y_tr).coef_ for a in alphas]
ridge_coefs, lasso_coefs = np.array(ridge_coefs), np.array(lasso_coefs)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for j in range(X_tr.shape[1]):
    ax1.semilogx(alphas, ridge_coefs[:, j])
    ax2.semilogx(alphas, lasso_coefs[:, j])
ax1.set_title("Ridge: Regularization Path (all shrink toward 0, never zero)")
ax2.set_title("Lasso: Regularization Path (features zeroed out)")
for ax in [ax1, ax2]: ax.set_xlabel("Lambda"); ax.set_ylabel("Coef"); ax.axhline(0, color="k", lw=0.5)
plt.tight_layout(); plt.savefig("/tmp/reg_paths.png", dpi=80); plt.show()

print("=== Lasso Sparsity at Different Lambda ===")
for a in [0.01, 0.1, 1.0, 10.0]:
    l = Lasso(alpha=a, max_iter=5000).fit(X_tr, y_tr)
    print(f"  alpha={a:6.2f}: {(l.coef_==0).sum():2d}/{X_tr.shape[1]} features zeroed out")

## 3. Logistic Regression

Logistic regression models log-odds: `log(P(y=1|x) / P(y=0|x)) = w^T x + b`

Equivalently: `P(y=1|x) = sigmoid(w^T x + b) = 1 / (1 + exp(-(w^T x + b)))`

**When to use**:
- Text classification (TF-IDF sparse features)
- Baseline for any binary classification problem
- When you need calibrated probabilities (fraud scoring, ad CTR)
- High-dimensional sparse features (one-hot encoding)

**C = 1/lambda**: Small C = strong regularization = simpler model. Default C=1.0.

In [ ]:
from sklearn.metrics import roc_auc_score

data = load_breast_cancer(); X_bc, y_bc = data.data, data.target
X_tr_bc, X_te_bc, y_tr_bc, y_te_bc = train_test_split(X_bc, y_bc, test_size=0.2, stratify=y_bc, random_state=42)
scaler_bc = StandardScaler()
X_tr_s = scaler_bc.fit_transform(X_tr_bc); X_te_s = scaler_bc.transform(X_te_bc)

print("=== Logistic Regression: Effect of C (C = 1/lambda) ===")
print(f"{"C":<8} {"Train AUC":>10} {"Test AUC":>10}")
for C in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]:
    lr = LogisticRegression(C=C, max_iter=1000, random_state=42).fit(X_tr_s, y_tr_bc)
    train_auc = roc_auc_score(y_tr_bc, lr.predict_proba(X_tr_s)[:,1])
    test_auc = roc_auc_score(y_te_bc, lr.predict_proba(X_te_s)[:,1])
    print(f"{C:<8} {train_auc:>10.4f} {test_auc:>10.4f}")

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| Normal equation | Closed-form OLS: w = (X^T X)^-1 X^T y; breaks when p > n |
| Ridge (L2) | Shrinks all coefficients; handles correlated features; never zeros |
| Lasso (L1) | Soft-thresholds to zero; automatic feature selection |
| ElasticNet | L1 + L2 mix; useful when features are correlated and you want sparsity |
| Logistic regression | Log-odds model; calibrated probabilities; linear decision boundary |
| C in LogReg | C = 1/lambda; small C = strong regularization |

### Common Pitfalls
- Using unscaled features with Ridge/Lasso (coefficients are not comparable)
- Using Lasso when features are highly correlated (arbitrarily drops one)
- Not cross-validating lambda (use RidgeCV / LassoCV)
- Treating logistic regression as always calibrated (recalibrate on imbalanced data)
